In [85]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.optimize import minimize
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint, uniform
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from mealpy.swarm_based import ACOR
import warnings
import joblib
import mlflow
import mlflow.sklearn
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal, List
import uvicorn
from sklearn.metrics import classification_report
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# --------------------- CONFIGURATION ---------------------
CONFIG = {
    # Data split
    'test_size': 0.20,
    'random_state': 42,
    
    # OOF generation
    'n_folds': 6,
    'inner_cv_folds': 6,
    'hpo_n_iter': 18,
    
    # Optimization splits
    'calib_ratio': 0.25,
    
    # ACO (mealpy)
    'n_ants': 40,
    'archive_size': 20,
    'max_iter': 100,
    'q': 0.1,
    'xi': 0.85,
    
    # Cost matrix
    'cost_fn': 5,   # False Negative cost
    'cost_fp': 1,   # False Positive cost
    
    # ANFIS
    'anfis_epochs': 500,
    'anfis_lr': 0.01,
}

In [86]:
print("--- Data Ingestion & Strict Splitting ---")

data = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
X = data.data
y = data.target.map({'good': 0, 'bad': 1}).values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG['test_size'], stratify=y, random_state=CONFIG['random_state']
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

--- Data Ingestion & Strict Splitting ---
Train: (800, 20), Test: (200, 20)


In [87]:

numerical_cols   = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'object']).columns.tolist()
cat_indices = list(range(len(numerical_cols), len(numerical_cols) + len(categorical_cols)))

# Stage 1: Robust scaling + pass categoricals through
prep_stage1 = ColumnTransformer([
    ('num', RobustScaler(), numerical_cols),
    ('cat', 'passthrough', categorical_cols)
])

# Stage 2: Identity cast + one-hot encoding (after SMOTENC)
prep_stage2 = ColumnTransformer([
    ('num_pass', StandardScaler(with_mean=False, with_std=False), list(range(len(numerical_cols)))),
    ('cat_ohe', OneHotEncoder(drop='first', sparse_output=False), cat_indices)
])


rf_base = RandomForestClassifier(random_state=CONFIG['random_state'])
rf_cal = CalibratedClassifierCV(rf_base, method='isotonic', cv=CONFIG['n_folds'])

svm_base = SVC(kernel='rbf', random_state=CONFIG['random_state'])
svm_cal = CalibratedClassifierCV(svm_base, method='sigmoid', cv=CONFIG['n_folds'])

mlp_base = MLPClassifier(solver='adam', max_iter=2000, early_stopping=True,
                         n_iter_no_change=18, random_state=CONFIG['random_state'])
mlp_cal = CalibratedClassifierCV(mlp_base, method='sigmoid', cv=CONFIG['n_folds'])

# Define full pipelines
pipelines = {
    'Random_Forest': ImbPipeline([
        ('prep1', clone(prep_stage1)),
        ('smote', SMOTENC(categorical_features=cat_indices, k_neighbors=5,
                          random_state=CONFIG['random_state'])),
        ('prep2', clone(prep_stage2)),
        ('clf', rf_cal)
    ]),
    'SVM': ImbPipeline([
        ('prep1', clone(prep_stage1)),
        ('smote', SMOTENC(categorical_features=cat_indices, k_neighbors=5,
                          random_state=CONFIG['random_state'])),
        ('prep2', clone(prep_stage2)),
        ('clf', svm_cal)
    ]),
    'MLP': ImbPipeline([
        ('prep1', clone(prep_stage1)),
        ('smote', SMOTENC(categorical_features=cat_indices, k_neighbors=5,
                          random_state=CONFIG['random_state'])),
        ('prep2', clone(prep_stage2)),
        ('clf', mlp_cal)
    ])
}

In [88]:
param_dist = {
    'Random_Forest': {
        'clf__estimator__n_estimators': randint(100, 400),
        'clf__estimator__max_depth': randint(4, 12),
        'clf__estimator__min_samples_leaf': randint(5, 15),
        'clf__estimator__max_features': uniform(0.1, 0.5)
    },
    'SVM': {
        'clf__estimator__C': loguniform(1e-3, 1e2),
        'clf__estimator__gamma': loguniform(1e-4, 1e0)
    },
    'MLP': {
        'clf__estimator__alpha': loguniform(1e-5, 1e-1),
        'clf__estimator__learning_rate_init': loguniform(5e-4, 1e-2),
        'clf__estimator__hidden_layer_sizes': [(32,), (64,), (128,), (64, 32)]
    }
}

In [89]:
# Cell 6 - MISSING
skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['random_state'])

oof_probs = np.zeros((len(X_train), len(pipelines)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train[train_idx]
    X_fold_val   = X_train.iloc[val_idx]

    for i, (name, pipeline) in enumerate(pipelines.items()):
        pipe = clone(pipeline)
        pipe.fit(X_fold_train, y_fold_train)
        oof_probs[val_idx, i] = pipe.predict_proba(X_fold_val)[:, 1]

trained_pipelines = {}
for i, (name, pipeline) in enumerate(pipelines.items()):
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

In [90]:
P_opt, P_calib, y_opt, y_calib = train_test_split(
    oof_probs, y_train, test_size=CONFIG['calib_ratio'],
    stratify=y_train, random_state=CONFIG['random_state']
)
y_opt = y_opt.astype(float)

In [91]:
def find_qp_weights(P, y):
    """Return optimal ensemble weights minimizing MSE via QP."""
    Q = 2 * P.T @ P
    c = -2 * (P.T @ y)
    
    n_models = P.shape[1]
    w_init = np.ones(n_models) / n_models
    bounds = [(0, 1)] * n_models
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
    
    res = minimize(
        lambda w: 0.5 * w @ Q @ w + c @ w,
        w_init, method='SLSQP', bounds=bounds, constraints=constraints,
        options={'ftol': 1e-9}
    )
    assert res.success, "QP solver failed"
    w = np.clip(res.x, 0, 1)
    w /= w.sum()
    return w

w_qp = find_qp_weights(P_opt, y_opt)
mse_qp = np.mean((P_opt @ w_qp - y_opt) ** 2)
print(f"QP weights: {w_qp}, MSE = {mse_qp:.6f}")

QP weights: [0.56917863 0.12164274 0.30917863], MSE = 0.161073


In [92]:
from mealpy.utils.space import FloatVar

# Objective function for mealpy
def aco_objective(w_raw):
    # Softmax mapping to simplex
    w = np.exp(w_raw - np.max(w_raw))
    w = w / w.sum()
    return np.mean((P_opt @ w - y_opt) ** 2)

# Define bounds as a list of FloatVar objects
bounds = [FloatVar(lb=-5, ub=5) for _ in range(P_opt.shape[1])]

# Problem definition
problem = {
    "obj_func": aco_objective,
    "bounds": bounds,
    "minmax": "min",
    "log_to": None
}

# ACOR model
model = ACOR.OriginalACOR(
    epoch=CONFIG['max_iter'],
    n_ants=CONFIG['n_ants'],
    n_sample=CONFIG['archive_size'],
    q=CONFIG['q'],
    xi=CONFIG['xi'],
    sigma=0.1
)

# Solve
best_agent = model.solve(problem)
w_aco_raw = best_agent.solution
best_fitness = best_agent.target.fitness

w_aco = np.exp(w_aco_raw - np.max(w_aco_raw))
w_aco = w_aco / w_aco.sum()

print(f"ACO weights: {w_aco}, MSE = {best_fitness:.6f}")
print(f"Sub‑optimality gap vs QP: {100 * (best_fitness - mse_qp) / mse_qp:.4f}%")

ACO weights: [0.56949796 0.10307775 0.32742429], MSE = 0.161080
Sub‑optimality gap vs QP: 0.0041%


In [119]:
def find_cost_optimal_threshold(y_true, y_probs, cost_fn=5, cost_fp=1):
    thresholds = np.sort(np.unique(y_probs))
    preds = y_probs[:, None] >= thresholds[None, :]
    fn = ((y_true[:, None] == 1) & (preds == 0)).sum(axis=0)
    fp = ((y_true[:, None] == 0) & (preds == 1)).sum(axis=0)
    costs = fn * cost_fn + fp * cost_fp
    best_idx = np.argmin(costs)
    return thresholds[best_idx], costs[best_idx]

t_aco, cost_aco = find_cost_optimal_threshold(y_calib, P_calib @ w_aco)
t_qp, cost_qp = find_cost_optimal_threshold(y_calib, P_calib @ w_qp)
print(f"ACO threshold = {t_aco:.4f} (min cost {cost_aco})")
print(f"QP threshold  = {t_qp:.4f} (min cost {cost_qp})")

ACO threshold = 0.1930 (min cost 91)
QP threshold  = 0.1919 (min cost 91)


In [120]:
print("\n=== Final Evaluation on Test Set ===")

# Generate test probabilities from each base model
P_test = np.zeros((X_test.shape[0], len(trained_pipelines)))
for i, (name, pipe) in enumerate(trained_pipelines.items()):
    P_test[:, i] = pipe.predict_proba(X_test)[:, 1]

# Ensemble predictions
pred_aco = P_test @ w_aco
pred_qp  = P_test @ w_qp

# Apply thresholds
binary_aco = (pred_aco >= t_aco).astype(int)
binary_qp  = (pred_qp  >= t_qp).astype(int)

def expected_cost(y_true, y_pred, cost_fn=5, cost_fp=1):
    cm = confusion_matrix(y_true, y_pred)
    return cm[1,0] * cost_fn + cm[0,1] * cost_fp

print(f"ACO - ROC-AUC: {roc_auc_score(y_test, pred_aco):.4f}, "
      f"Cost: {expected_cost(y_test, binary_aco)}")
print(f"QP  - ROC-AUC: {roc_auc_score(y_test, pred_qp):.4f}, "
      f"Cost: {expected_cost(y_test, binary_qp)}")

print("\n=== Classification Report: ACO ===")
print(classification_report(y_test, binary_aco))

print("\n=== Classification Report: QP ===")
print(classification_report(y_test, binary_qp))


=== Final Evaluation on Test Set ===
ACO - ROC-AUC: 0.8229, Cost: 111
QP  - ROC-AUC: 0.8227, Cost: 111

=== Classification Report: ACO ===
              precision    recall  f1-score   support

           0       0.89      0.53      0.66       140
           1       0.44      0.85      0.58        60

    accuracy                           0.62       200
   macro avg       0.66      0.69      0.62       200
weighted avg       0.75      0.62      0.64       200


=== Classification Report: QP ===
              precision    recall  f1-score   support

           0       0.89      0.53      0.66       140
           1       0.44      0.85      0.58        60

    accuracy                           0.62       200
   macro avg       0.66      0.69      0.62       200
weighted avg       0.75      0.62      0.64       200



In [95]:
def inv_sigmoid(y, min_val, max_val):
    """Inverse sigmoid mapping from bounded to unbounded space."""
    p = (y - min_val) / (max_val - min_val)
    return np.log(p / (1 - p))

    """Gaussian membership function with sigmoid reparameterization."""
class GaussianMF(nn.Module):
    def __init__(self, centers, sigmas):
        super().__init__()
        self.min_c, self.max_c = 0.0, 1.0
        self.min_s, self.max_s = 0.05, 0.3

        raw_centers = [inv_sigmoid(c, self.min_c, self.max_c) for c in centers]
        self._raw_centers = nn.Parameter(torch.tensor(raw_centers, dtype=torch.float32))

        raw_sigmas = [inv_sigmoid(s, self.min_s, self.max_s) for s in sigmas]
        self._raw_sigmas = nn.Parameter(torch.tensor(raw_sigmas, dtype=torch.float32))

    @property
    def centers(self):
        return self.min_c + (self.max_c - self.min_c) * torch.sigmoid(self._raw_centers)

    @property
    def sigmas(self):
        return self.min_s + (self.max_s - self.min_s) * torch.sigmoid(self._raw_sigmas)

    def forward(self, x):
        return torch.exp(-0.5 * ((x - self.centers) / self.sigmas) ** 2)

class ANFISModel(nn.Module):
    """PyTorch module for the ANFIS network."""
    def __init__(self, n_rules=3):
        super().__init__()
        self.fuzzify = GaussianMF([0.1, 0.5, 0.9], [0.2, 0.2, 0.2])
        self.a = nn.Parameter(torch.tensor([0.0, 0.5, 1.0], dtype=torch.float32))
        self.b = nn.Parameter(torch.zeros(3, dtype=torch.float32))

    def forward(self, x):
        w = self.fuzzify(x)
        w_norm = w / (w.sum(dim=1, keepdim=True) + 1e-8)
        f = x * self.a + self.b
        return (w_norm * f).sum(dim=1)

class ANFIS(BaseEstimator, TransformerMixin):
    """Adaptive Neuro‑Fuzzy Inference System as a scikit‑learn estimator."""
    def __init__(self, n_rules=3, lr=0.01, epochs=500, random_state=42):
        self.n_rules = n_rules
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        self.model_ = ANFISModel(n_rules=self.n_rules)
        optimizer = optim.Adam(self.model_.parameters(), lr=self.lr)
        X_t = torch.tensor(X, dtype=torch.float32).view(-1, 1)
        y_t = torch.tensor(y, dtype=torch.float32)

        self.model_.train()
        for _ in range(self.epochs):
            optimizer.zero_grad()
            loss = nn.MSELoss()(self.model_(X_t), y_t)
            loss.backward()
            optimizer.step()
        return self

    def transform(self, X):
        self.model_.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32).view(-1, 1)
            return self.model_(X_t).numpy()

    def predict(self, X):
        return self.transform(X)


In [96]:
anfis = ANFIS(
    n_rules=3,
    lr=CONFIG['anfis_lr'],
    epochs=CONFIG['anfis_epochs'],
    random_state=CONFIG['random_state']
)
anfis.fit((oof_probs @ w_aco).reshape(-1, 1), y_train)  # clean, no leakage

,n_rules,3
,lr,0.01
,epochs,500
,random_state,42


In [97]:
# Save ANFIS model separately (PyTorch state_dict)
anfis_state_dict = anfis.model_.state_dict()
anfis_params = {
    'n_rules': anfis.n_rules,
    'lr': anfis.lr,
    'epochs': anfis.epochs,
    'random_state': anfis.random_state
}
torch.save(anfis_state_dict, 'anfis_state_dict.pth')

col_trans = list(trained_pipelines.values())[0].named_steps['prep2']
ohe = {name: trans for name, trans, _ in col_trans.transformers_}['cat_ohe']

# Save other artifacts
model_artifacts = {
    'trained_pipelines': trained_pipelines,
    'w_aco': w_aco,
    't_aco': t_aco,
    'anfis_params': anfis_params,
    'feature_names': list(X.columns),
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols,
    'cat_indices': cat_indices,
    'ohe_categories': {
        col: cats.tolist()
        for col, cats in zip(categorical_cols, ohe.categories_)
    },
    'numerical_ranges': {
        col: (float(X_train[col].min()), float(X_train[col].max()))
        for col in numerical_cols
    }
}
joblib.dump(model_artifacts, 'credit_risk_model.pkl')
print("Model saved to credit_risk_model.pkl and anfis_state_dict.pth")

Model saved to credit_risk_model.pkl and anfis_state_dict.pth


In [121]:
# Start MLflow run
mlflow.set_experiment("Credit Risk Assessment")
with mlflow.start_run(run_name="ACO_Ensemble") as run:
    # Log parameters
    mlflow.log_params(CONFIG)
    
    # Log metrics
    mlflow.log_metric("test_roc_auc", roc_auc_score(y_test, pred_aco))
    mlflow.log_metric("test_cost", expected_cost(y_test, binary_aco))
    
    # Log model (using the ACO ensemble)
    # We can log the full pipeline as a custom model or log the artifacts
    mlflow.sklearn.log_model(
        sk_model=anfis,  # Or a custom wrapper
        name="anfis_model",
        registered_model_name="CreditRiskANFIS"
    )
    
    # Log artifacts
    mlflow.log_artifact("credit_risk_model.pkl")
    
    print(f"MLflow run ID: {run.info.run_id}")

2026/03/29 16:20:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/29 16:21:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


MLflow run ID: 834a664e9b0e46f58ad920cf57399572


Registered model 'CreditRiskANFIS' already exists. Creating a new version of this model...
Created version '9' of model 'CreditRiskANFIS'.


In [99]:
import gradio as gr
import joblib
import torch
import numpy as np
import pandas as pd

# Load artifacts
artifacts = joblib.load('credit_risk_model.pkl')
trained_pipelines = artifacts['trained_pipelines']
w_aco = artifacts['w_aco']
t_aco = artifacts['t_aco']
anfis_params = artifacts['anfis_params']
feature_names = artifacts['feature_names']
numerical_cols = artifacts['numerical_cols']
categorical_cols = artifacts['categorical_cols']
ohe_categories = artifacts['ohe_categories']
numerical_ranges = artifacts['numerical_ranges']

# Load ANFIS model
anfis_state_dict = torch.load('anfis_state_dict.pth', map_location='cpu')
anfis_model = ANFISModel(n_rules=anfis_params['n_rules'])
anfis_model.load_state_dict(anfis_state_dict)
anfis_model.eval()

# Define prediction function
def predict_credit_risk(*args):
    input_dict = {name: val for name, val in zip(feature_names, args)}
    df = pd.DataFrame([input_dict])

    for col in numerical_cols:
        df[col] = pd.to_numeric(df[col])

    P = np.zeros((1, len(trained_pipelines)))
    for i, (name, pipeline) in enumerate(trained_pipelines.items()):
        P[0, i] = pipeline.predict_proba(df)[0, 1]

    final_prob = float((P @ w_aco)[0])
    decision = "❌ REJECT (High Risk)" if final_prob >= t_aco else "✅ APPROVE (Low Risk)"

    with torch.no_grad():
        x_t = torch.tensor([[final_prob]], dtype=torch.float32)
        linguistic = float(anfis_model(x_t).item())

    return decision, f"{final_prob:.4f}", f"{linguistic:.3f}"

# Build inputs dynamically from saved artifacts — no hardcoding
inputs = []
for col in feature_names:
    if col in numerical_ranges:
        min_val, max_val = numerical_ranges[col]
        inputs.append(gr.Slider(minimum=min_val, maximum=max_val, step=1 if isinstance(min_val, int) else 0.01, label=col))
    else:
        inputs.append(gr.Dropdown(choices=ohe_categories[col], label=col))

outputs = [
    gr.Textbox(label="Financial Decision"),
    gr.Textbox(label="Risk Probability"),
    gr.Textbox(label="Linguistic Assessment (ANFIS)")
]

demo = gr.Interface(
    fn=predict_credit_risk,
    inputs=inputs,
    outputs=outputs,
    title="🏦 Advanced Credit Risk Analyzer",
    description="Asymmetric Cost-Sensitive Ensemble (ACO + ANFIS). Decision threshold optimized for 5:1 False Negative penalty.",
)

demo.launch(share=True, theme="soft")

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://65297e684310246126.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [100]:
print(t_aco)

0.192983833769545
